## PyINE prompt result viewer

This notebook shows how to use the framework's prompt result database to view prompt results in an interactive fashion.

The code below will rely on the default prompt result database (via `pyine.prompts.get_framework_db_path()`), and it will display its content (according to some optional filters) using ipywidgets.

In [ ]:
import html
import json

import ipywidgets as widgets
from IPython.display import HTML, display

import pyine.prompts
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()  # loads dotenv variables, seeds, sets up logging, etc.
result_db = pyine.prompts.get_framework_db()


def _escape(
    text: str,
) -> str:
    """HTML-escape and keep newlines."""
    return html.escape(text).replace("\n", "<br>")


def _collapsible_block(
    title: str,
    raw_text: str,
    *,
    threshold: int = 600,
    max_height_px: int = 420,
) -> str:
    """Return an HTML block that is collapsible if the text is long.

    Args:
        title: Section title shown in the summary.
        raw_text: Unescaped content to display inside the block.
        threshold: If len(raw_text) > threshold, the block is collapsed by default.
        max_height_px: Max visible height for the content (scrolls if larger).

    Returns:
        An HTML snippet as a string.
    """
    is_long = len(raw_text) > threshold
    escaped = _escape(raw_text)
    open_attr = "" if is_long else " open"
    summary_suffix = f" ({len(raw_text)} chars)"
    return f"""\
        <div style="margin-bottom:12px">
          <details{open_attr}>
            <summary style="cursor:pointer; font-weight:600">{html.escape(title)}{summary_suffix}</summary>
            <div style="white-space:pre-wrap; border:1px solid #444; padding:8px; border-radius:8px; margin-top:8px; max-height:{max_height_px}px; overflow:auto">{escaped}</div>
          </details>
        </div>
        """


def view_llm_results(
    records: list["pyine.prompts.PromptResultRecord"],
) -> None:
    """Notebook viewer with next/prev controls for LLM results.

    Args:
        records: List of records to display.
    """
    if not records:
        display(HTML("<b>No records.</b>"))
        return

    idx_slider = widgets.IntSlider(value=0, min=0, max=len(records) - 1, step=1, description="idx")
    prev_btn = widgets.Button(description="◀ Prev")
    next_btn = widgets.Button(description="Next ▶")
    out = widgets.Output()

    def render(idx: int) -> None:
        rec = records[idx]
        small_blocks = [
            ("Identifier", f"id={rec.identifier}, group={rec.group}"),
            ("Prompt, version", f"name={rec.prompt_name}. version={rec.prompt_version}"),
            ("Created at", rec.creation_meta.created_at.strftime("%Y-%m-%d %H:%M:%S")),
            ("Tags", str(rec.tags)),
        ]
        html_parts: list[str] = [
            f"""<div style="font-family: ui-monospace, SFMono-Regular, Menlo, monospace; line-height:1.35">
            <div style="margin-bottom:12px"><b>Index:</b> {idx+1}/{len(records)}</div>
            """
        ]
        for title, text in small_blocks:
            html_parts.append(
                f"<div style='margin-bottom:12px'><b>{html.escape(title)}</b>"
                f"<div style='white-space:pre-wrap; border:1px solid #444; padding:8px; border-radius:8px'>{_escape(text)}</div></div>"
            )
        html_parts.append(_collapsible_block("Response", rec.result, threshold=800))
        html_parts.append(_collapsible_block("Prompt", rec.prompt, threshold=600))
        html_parts.append(_collapsible_block("Metadata", json.dumps(rec.meta, indent=2), threshold=400))
        html_parts.append("</div>")
        page_html = "\n".join(html_parts)
        out.clear_output(wait=True)
        with out:
            display(HTML(page_html))

    def on_prev(_: widgets.Button) -> None:
        if idx_slider.value == idx_slider.min:
            idx_slider.value = idx_slider.max
        else:
            idx_slider.value = idx_slider.value - 1

    def on_next(_: widgets.Button) -> None:
        if idx_slider.value == idx_slider.max:
            idx_slider.value = idx_slider.min
        else:
            idx_slider.value = idx_slider.value + 1

    def on_change(change: dict) -> None:
        if change["name"] == "value":
            render(change["new"])

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    idx_slider.observe(on_change)

    controls = widgets.HBox([prev_btn, next_btn, idx_slider])
    display(controls, out)
    render(idx_slider.value)

In [ ]:
available_prompt_names = result_db.list_prompt_names()
available_groups = result_db.list_groups()
available_identifiers = result_db.list_identifiers()

groups_str = f", namely: {available_groups}" if len(available_groups) < 20 else ""
identifiers_str = f", namely: {available_identifiers}" if len(available_identifiers) < 20 else ""

print("database contains prompting results for:")
print(f"\t{len(available_prompt_names)} prompt type(s), namely: {available_prompt_names}")
print(f"\t{len(available_groups)} group(s){groups_str}")
print(f"\t{len(available_identifiers)} identifier(s){identifiers_str}")

In [ ]:
# for the visualization: target a specific prompt name, and display all results for it
target_prompt_name = "hints/docs"  # MODIFY ME IF NEEDED!
target_prompt_version = None  # MODIFY ME IF NEEDED!
tag_filter_rule = None  # MODIFY ME IF NEEDED!
max_result_age = None  # MODIFY ME IF NEEDED!

if target_prompt_name not in available_prompt_names:
    raise ValueError(f"prompt name '{target_prompt_name}' not found in database")

found_records = result_db.get_by_prompt_name(
    prompt_name=target_prompt_name,
    prompt_version=target_prompt_version,
    tag_filter_rule=tag_filter_rule,
    max_result_age=max_result_age,
)

view_llm_results(found_records)